# Interactive Query Testing

In [16]:
import sys, os, yaml, torch, pickle
import pandas as pd
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer, CrossEncoder

# Setup project root
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.esci.sparse_retrievers import SPLADEFast, BM25Fast

# Load Config
with open(project_root / "configs" / "esci.yaml") as f:
    cfg = yaml.safe_load(f)

In [17]:
print("Loading artifacts (Catalog, Embeddings, Pairs)...")
from src.esci.artifacts import load_artifacts
from src.esci.faiss_utils import build_faiss_index

# Loading Artifacts
prod_emb, qry_emb, prod_df, qry_df, pair_df = load_artifacts("us", "test", cfg)

print("Preparing Dictionaries...")
prod_df_unique = prod_df.drop_duplicates(subset=["product_id"])
prod_dict = prod_df_unique.set_index("product_id")[["product_title", "product_text_dense"]].to_dict('index')

# Mapping FAISS need to be the same as prod_emb
pid_map = prod_df["product_id"].tolist()

print("Loading Dense model (Matryoshka)...")
matryoshka_path = str(Path(cfg["paths"]["matryoshka_dir"]) / "us")
dense_model = SentenceTransformer(matryoshka_path, device="cuda" if torch.cuda.is_available() else "cpu")

print("Building FAISS index on the fly (64 dimensions)...")

faiss_index = build_faiss_index(prod_emb)

print("Loading SPLADE...")
splade_data = torch.load(project_root / "artifacts" / "systemA" / "splade_data.pt", map_location="cpu")
splade = SPLADEFast(cfg["sparse"]["splade_model"])
splade.doc_matrix = splade_data["doc_matrix"].to(splade.device)
splade.pid_list = splade_data["pid_list"]

print("Loading BM25...")
with open(project_root / "artifacts" / "systemA" / "bm25_retriever.pkl", "rb") as f:
    bm25 = pickle.load(f)

print("Loading Cross-Encoder (Reranker)...")
reranker = CrossEncoder(cfg["cross_encoder_model"], device="cuda" if torch.cuda.is_available() else "cpu")

print("✅ All systems go!")

Loading artifacts (Catalog, Embeddings, Pairs)...
Preparing Dictionaries...
Loading Dense model (Matryoshka)...
Building FAISS index on the fly (64 dimensions)...
Loading SPLADE...


C:\Users\aminl\anaconda3\envs\pytorch-gpu\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Loading BM25...
Loading Cross-Encoder (Reranker)...
✅ All systems go!


In [18]:
def reciprocal_rank_fusion(dense_hits, sparse_hits, bm25_hits, k=60, weights=(1.0, 0.7, 0.3)):
    """Fuses results using RRF with optimized weights."""
    scores = {}
    
    def add_to_rrf(hits, weight):
        for rank, (pid, score) in enumerate(hits):
            if pid not in scores:
                scores[pid] = 0.0
            scores[pid] += weight / (k + rank + 1)
            
    add_to_rrf(dense_hits, weights[0])
    add_to_rrf(sparse_hits, weights[1])
    add_to_rrf(bm25_hits, weights[2])
    
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def search(query: str, top_k=10, candidates_to_retrieve=200, candidates_to_rerank=50):
    """Full Pipeline: Query -> Hybrid RRF -> Cross-Encoder"""
    
    # 1. Dense Encoding (Matryoshka 64d)
    q_emb = dense_model.encode(["Represent this sentence for searching relevant passages: " + query], normalize_embeddings=True)
    
    # FAISS Search (Native)
    D, I = faiss_index.search(q_emb, candidates_to_retrieve)
    dense_hits = [(pid_map[idx], float(score)) for idx, score in zip(I[0], D[0]) if idx != -1]
    
    # 2. Sparse (SPLADE) & Lexical (BM25) Search
    splade_raw = splade.score_topk(query, top_k=candidates_to_retrieve)
    sparse_hits = [(splade.pid_list[idx], float(score)) for idx, score in splade_raw]
    bm25_hits = bm25.search(query, top_k=candidates_to_retrieve)
    
    # 3. RRF Fusion
    fused_candidates = reciprocal_rank_fusion(dense_hits, sparse_hits, bm25_hits)
    top_pids = [pid for pid, score in fused_candidates[:candidates_to_rerank]]
    
    # 4. Reranking with Cross-Encoder
    cross_inp = [[query, prod_dict[pid]["product_text_dense"]] for pid in top_pids]
    rerank_scores = reranker.predict(cross_inp)
    
    # 5. Formatting the results
    results = []
    for idx, pid in enumerate(top_pids):
        results.append({
            "Rerank Score": round(float(rerank_scores[idx]), 3),
            "Product Title": prod_dict[pid]["product_title"]
        })
        
    final_df = pd.DataFrame(results).sort_values("Rerank Score", ascending=False).head(top_k)
    
    # Adjust Pandas display to read long titles
    pd.set_option('display.max_colwidth', None)
    return final_df.reset_index(drop=True)

In [19]:
search("noise cancelling wireless headphones over ear")

,Rerank Score,Product Title
0,0.997,"Bose Noise Cancelling Headphones 700, Bluetooth, Over-Ear Wireless Headphones with Built-In Microphone for Clear Calls & Alexa Voice Control, Black"
1,0.997,"Audonia E7 Active Noise Cancelling Headphones, Over Ear Bluetooth Headphones with Mic, Hi-Def Audio, Deep Bass, Memory Foam Ear Cups Wireless Headphones with 20H Playtime for Travel, Home - Black"
2,0.997,"Active Noise Cancelling Headphones - SHIVR 3D Immersive Audio Wireless Over Ear Bluetooth Headset with Microphone, Built-in Gyroscope Smart Play/Pause, Ambient Sound Mode for Travel Work TV PC Phones"
3,0.996,"MOVSSOU E7 Active Noise Cancelling Headphones Bluetooth Headphones Wireless Headphones Over Ear with Microphone Deep Bass, Comfortable Protein Earpads, 30 Hours Playtime for Travel/Work, Black"
4,0.996,"Active Noise Cancelling Headphones, TaoTronics Bluetooth Headphones Over Ear Wireless Headphones 40H Playtime Type-C Fast Charging Bluetooth 5.0 CVC 8.0 Mic for Online Class Black"
5,0.996,"Master & Dynamic MW65 Active Noise-Cancelling (Anc) Wireless Headphones – Bluetooth Over-Ear Headphones with Mic, Silver Metal/ Brown Leather"
6,0.995,COWIN E7 PRO [Upgraded] Active Noise Cancelling Headphones Bluetooth Headphones with Microphone/Deep Bass Wireless Headphones Over Ear 30H Playtime for Travel/Work/TV/Computer/Cellphone - Black
7,0.995,"Bose QuietComfort 35 II Noise Cancelling Bluetooth Headphones— Wireless, Over Ear Headphones with Built in Microphone and Alexa Voice Control, Silver"
8,0.995,"Beats Studio3 Wireless Noise Cancelling Over-Ear Headphones - Apple W1 Headphone Chip, Class 1 Bluetooth, Active Noise Cancelling, 22 Hours of Listening Time - Sand Dune"
9,0.994,Sony WH1000XM3 Noise Cancelling Headphones : Wireless Bluetooth Over the Ear Headset – Silver (2018 Version)


In [20]:
search("blue running shoes size 10")

,Rerank Score,Product Title
0,0.585,"Saucony Women's Cohesion 10 Running Shoe, Navy Blue, 10.5 Medium US"
1,0.461,"Nike Men's Track & Field Shoes, Multicolour Obsidian Mist Blue Lagoon Half Blue 400, Womens 10"
2,0.389,"adidas Originals Women's U_Path X W Running Shoe, White/Real Blue/Night Metallic, 10 M US"
3,0.323,Nike Air Zoom Pegasus 36 Men's Running Shoe Racer Blue/Black-White Size 11.0
4,0.322,"ASICS Women's Gel-Cumulus 20 Running Shoes, 10, Azure/Blue Print"
5,0.315,"adidas Men's Adizero Tempo 9 Running Shoe, hi-res Blue/Silver Metallic/Legend Ink, 7 M US"
6,0.291,"New Balance Women's FuelCell Propel V1 Running Shoe, Blue/White, 10 M US"
7,0.226,"adidas Men's Ultraboost 20 Primeblue Running Shoe, White/Sharp Blue/true orange, 13 M US"
8,0.211,"Brooks Women's Adrenaline GTS 19, Blue/Aqua, 8 B"
9,0.202,Brooks Ghost 11 Grey/Blue/Silver 8 4E - Extra Wide


In [21]:
search("breville espresso machine BES870XL")

,Rerank Score,Product Title
0,0.991,Breville BES870XL Barista Express Espresso Machine (Renewed)
1,0.982,"Breville BES870XL Barista Express Espresso Machine, Brushed Stainless Steel"
2,0.973,Breville BES860XL Barista Express Espresso Machine with Grinder
3,0.970,"Breville BES870BSXL the Barista Express, Espresso Machine, Black Sesame"
4,0.902,Breville Barista Touch Semi-Automatic Touchscreen Espresso Machine Bundle w/Extra ClaroSwiss Filter Included - BES880
5,0.834,"Breville BES840XL Infuser Espresso Machine, Brushed Stainless Steel"
6,0.806,"Breville BES880BSS Barista Touch Espresso Machine, Brushed Stainless Steel"
7,0.803,"Breville BES980XL Oracle Espresso Machine, Brushed Stainless Steel"
8,0.795,"Breville BES920XL Dual Boiler Espresso Machine, Brushed Stainless Steel"
9,0.751,"MacMaxe ULKA Model E Type EFP5 – Solenoid Vibratory Water Pump – 2/1 min ON/OFF, 120V 60Hz 52W – For Breville Espresso Machines"
